<a href="https://colab.research.google.com/github/tryan01/gravitational-waves/blob/main/microquant_v1_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import torch
import pandas as pd
import numpy as np
import gcsfs
from google.colab import auth
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict

# 1. Authenticate Colab
print("Authenticating with Google Cloud...")
auth.authenticate_user()

# 2. Connect to Bucket
fs = gcsfs.GCSFileSystem()
bucket_path = 'tryanheider-kalshi-bucket/kalshi_dense_state_v2'

print(f"Scanning bucket: gs://{bucket_path}...")
all_dense_files = fs.glob(f"{bucket_path}/*.parquet")
all_dense_files = [f"gs://{f}" for f in all_dense_files]

if not all_dense_files:
    raise FileNotFoundError("No parquet files found in the bucket. Check your path.")

# 3. Pair Markets to Prevent Leakage
market_groups = {}
for file_path in all_dense_files:
    # Example filename: dense_KXMLBGAME-26APR151840KCDET-KC.parquet
    filename = os.path.basename(file_path)
    base_game = filename.split('-')[1] # Extracts '26APR151840KCDET'

    if base_game not in market_groups:
        market_groups[base_game] = []
    market_groups[base_game].append(file_path)

unique_games = list(market_groups.keys())
print(f"Found {len(unique_games)} unique MLB games.")

# Split 80/20 strictly by GAME
train_games, val_games = train_test_split(unique_games, test_size=0.2, random_state=42)

train_files = [f for game in train_games for f in market_groups[game]]
val_files = [f for game in val_games for f in market_groups[game]]

print(f"Train Set: {len(train_games)} games ({len(train_files)} individual tickers)")
print(f"Val Set: {len(val_games)} games ({len(val_files)} individual tickers)")

Authenticating with Google Cloud...
Scanning bucket: gs://tryanheider-kalshi-bucket/kalshi_dense_state_v2...
Found 52 unique MLB games.
Train Set: 41 games (82 individual tickers)
Val Set: 11 games (22 individual tickers)


In [9]:
# ---THE DATASET CLASS ---
class KalshiDenseDataset(Dataset):
    def __init__(self, file_paths: List[str], seq_len: int = 60, forward_look: int = 30, threshold: float = 0.015, stride: int = 5, inference_mode: bool = False):
        self.seq_len = seq_len
        self.forward_look = forward_look
        self.threshold = threshold
        self.stride = stride
        self.inference_mode = inference_mode # The toggle for inference.ipynb

        self.tensors = []
        self.labels = []
        self.timestamps = []
        self.tickers = [] # Tracking tickers for the Simulator
        self.audit_log = {'class_0': 0, 'class_1': 0, 'class_2': 0, 'nan_drops': 0}

        # Explicitly define columns that require log1p normalization
        self.vol_cols = [f'bid_vol_{i}' for i in range(1, 6)] + \
                        [f'ask_vol_{i}' for i in range(1, 6)] + \
                        ['taker_vol_yes', 'taker_vol_no']

        print(f"Loading {len(file_paths)} dense files into RAM (Inference Mode: {inference_mode})...")
        for f in file_paths:
            self._process_file(f)

        print("\n=== FINAL ENGINEERING AUDIT ===")
        print(f"Total valid {seq_len}-step sequences: {len(self.tensors)}")
        if not self.inference_mode:
            print(f"Distribution -> DOWN: {self.audit_log['class_0']} | FLAT: {self.audit_log['class_1']} | UP: {self.audit_log['class_2']}")
        print(f"Dropped due to NaNs: {self.audit_log['nan_drops']}")

    def _process_file(self, file_path: str):
        # Extract the ticker name from the filepath for the simulator
        ticker = os.path.basename(file_path).replace('dense_', '').replace('.parquet', '')

        # Read directly from GCS using pandas
        df = pd.read_parquet(file_path)
        if df.empty:
            return

        # 1. Target Math (Must be done BEFORE log normalization)
        df['micro_price'] = (df['ask_vol_1'] * df['bid_px_1'] + df['bid_vol_1'] * df['ask_px_1']) / (df['bid_vol_1'] + df['ask_vol_1'] + 1e-8)
        df['smoothed_target'] = df['micro_price'].rolling(10).mean().shift(-self.forward_look)

        # 2. Log1p Normalization
        for col in self.vol_cols:
            df[col] = np.log1p(df[col].values)

        # 3. Define Feature Vector (Drop target metrics and pure ts)
        feature_cols = [c for c in df.columns if c not in ['ts', 'micro_price', 'smoothed_target']]

        # 4. Extract Sliding Windows
        for i in range(0, len(df) - self.seq_len - self.forward_look, self.stride):
            window = df.iloc[i : i + self.seq_len][feature_cols].values

            # Labeling is strictly based on the final step of the window vs the future
            current_micro = df.iloc[i + self.seq_len - 1]['micro_price']
            future_micro = df.iloc[i + self.seq_len - 1]['smoothed_target']
            target_px_move = future_micro - current_micro

            trigger_ts = df.iloc[i + self.seq_len - 1]['ts']

            if np.isnan(window).any() or np.isnan(target_px_move):
                self.audit_log['nan_drops'] += 1
                continue

            # Classify: 0 (Down), 1 (Flat), 2 (Up)
            label = 1 # Default flat
            if target_px_move > self.threshold:
                label = 2
                self.audit_log['class_2'] += 1
            elif target_px_move < -self.threshold:
                label = 0
                self.audit_log['class_0'] += 1
            else:
                self.audit_log['class_1'] += 1

            self.tensors.append(torch.tensor(window, dtype=torch.float32))
            self.labels.append(label)
            self.timestamps.append(trigger_ts)
            self.tickers.append(ticker) # Store ticker for inference

    def __len__(self):
        return len(self.tensors)

    def __getitem__(self, idx):
        if self.inference_mode:
            # CONTRACT: Yields (ticker, ts, local_recv_ts, X_tensor)
            snapshot_time = self.timestamps[idx]
            return self.tickers[idx], snapshot_time, snapshot_time, self.tensors[idx]
        else:
            # ORIGINAL: Yields (X_tensor, y_label, timestamp) for training
            return self.tensors[idx], self.labels[idx], self.timestamps[idx]

# --- Instantiate DataLoaders ---
print("\nBuilding Training Dataset...")
train_dataset = KalshiDenseDataset(train_files, seq_len=60, forward_look=30, threshold=0.02, stride=5, inference_mode=False)

print("\nBuilding Validation Dataset...")
val_dataset = KalshiDenseDataset(val_files, seq_len=60, forward_look=30, threshold=0.02, stride=5, inference_mode=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"\nDataLoader Ready. Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


Building Training Dataset...
Loading 82 dense files into RAM (Inference Mode: False)...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 234684
Distribution -> DOWN: 12525 | FLAT: 209847 | UP: 12312
Dropped due to NaNs: 0

Building Validation Dataset...
Loading 22 dense files into RAM (Inference Mode: False)...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 62964
Distribution -> DOWN: 3507 | FLAT: 56014 | UP: 3443
Dropped due to NaNs: 0

DataLoader Ready. Train batches: 3667 | Val batches: 984


In [7]:
import pandas as pd
import gcsfs

# Connect to your bucket
fs = gcsfs.GCSFileSystem()
bucket_path = 'tryanheider-kalshi-bucket/kalshi_dense_state_v2'
files = fs.glob(f"{bucket_path}/*.parquet")

# Read just the first file
df = pd.read_parquet(f"gs://{files[0]}")

print("=== DENSE TENSOR FORENSICS ===")
print(f"File: {files[0]}")
print(f"Total Rows: {len(df)}")
# FIX: Look at the raw bid price, which is actually saved in the file
print(f"Unique values in bid_px_1: {df['bid_px_1'].nunique()}")
print(f"Number of zeroes in bid_vol_1: {(df['bid_vol_1'] == 0).sum()}")
print("\nFirst 5 rows of orderbook features:")
print(df[['ts', 'bid_px_1', 'bid_vol_1', 'ask_px_1', 'ask_vol_1']].head())

=== DENSE TENSOR FORENSICS ===
File: tryanheider-kalshi-bucket/kalshi_dense_state_v2/dense_KXMLBGAME-26APR151840CHCPHI-CHC.parquet
Total Rows: 14400
Unique values in bid_px_1: 67
Number of zeroes in bid_vol_1: 1

First 5 rows of orderbook features:
             ts  bid_px_1  bid_vol_1  ask_px_1  ask_vol_1
0  1.776288e+09      0.43      939.0      0.45    94750.0
1  1.776288e+09      0.43      939.0      0.45    94750.0
2  1.776288e+09      0.43      939.0      0.45    94750.0
3  1.776288e+09      0.43      939.0      0.45    94750.0
4  1.776288e+09      0.43      939.0      0.45    94750.0


In [10]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe: Tensor = torch.zeros(max_len, d_model)
        position: Tensor = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term: Tensor = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Shape: (1, max_len, d_model) for batch_first=True
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:, :x.size(1), :]
        return x

class KalshiTransformer(nn.Module):
    def __init__(self, feature_dim: int = 23, d_model: int = 64, nhead: int = 4, num_layers: int = 2, num_classes: int = 3):
        super().__init__()
        self.d_model: int = d_model

        # 1. Input Projection: Map 23 raw features to 64-dim embedding
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # 2. Transformer Encoder (batch_first=True ensures shape is [batch, seq, feature])
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes)
        )

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, feature_dim]
        Returns:
            logits: Tensor, shape [batch_size, num_classes]
        """
        # Project and encode
        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # Pass through attention layers
        x = self.transformer_encoder(x)

        # Extract the final time-step representation for sequence classification
        final_step: Tensor = x[:, -1, :]

        logits: Tensor = self.classifier(final_step)
        return logits



class FocalLoss(nn.Module):
    def __init__(self, alpha: Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha: Tensor = alpha
        self.gamma: float = gamma

    def forward(self, inputs: Tensor, targets: Tensor) -> Tensor:
        """
        Args:
            inputs: [batch_size, num_classes] (raw logits)
            targets: [batch_size] (integer class indices)
        """
        ce_loss: Tensor = F.cross_entropy(inputs, targets, reduction='none')
        pt: Tensor = torch.exp(-ce_loss)

        # Gather the alpha weights for the target classes
        alpha_t: Tensor = self.alpha.gather(0, targets)

        # Apply the focal equation
        focal_loss: Tensor = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [11]:
from tqdm import tqdm

# --- Setup Devices and Hyperparameters ---
device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model: nn.Module = KalshiTransformer(feature_dim=23, d_model=64, nhead=4, num_layers=2).to(device)
optimizer: torch.optim.Optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Calculate empirical Alpha weights based on your Train distribution
# Inverse frequency weighting: total_samples / (num_classes * class_samples)
class_counts = torch.tensor([5084, 87474, 4750], dtype=torch.float32)
alpha_weights: Tensor = class_counts.sum() / (3.0 * class_counts)
alpha_weights = alpha_weights.to(device)

criterion: nn.Module = FocalLoss(alpha=alpha_weights, gamma=2.0)

# --- Training Loop ---
epochs: int = 10
best_val_loss: float = float('inf')

for epoch in range(epochs):
    model.train()
    running_loss: float = 0.0
    correct: int = 0
    total: int = 0

    # Progress bar wrapping the training loader
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    for X_batch, Y_batch, _ in train_bar:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        optimizer.zero_grad()
        logits: Tensor = model(X_batch)
        loss: Tensor = criterion(logits, Y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Prevent explosions
        optimizer.step()

        running_loss += loss.item()

        # Compute basic accuracy
        preds: Tensor = torch.argmax(logits, dim=1)
        correct += (preds == Y_batch).sum().item()
        total += Y_batch.size(0)

        # Update progress bar metrics
        train_bar.set_postfix({'Loss': f"{loss.item():.4f}"})

    train_acc: float = correct / total

    # --- Validation ---
    model.eval()
    val_loss: float = 0.0
    val_correct: int = 0
    val_total: int = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val  ]")
        for X_batch, Y_batch, _ in val_bar:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

            logits: Tensor = model(X_batch)
            loss: Tensor = criterion(logits, Y_batch)

            val_loss += loss.item()
            preds: Tensor = torch.argmax(logits, dim=1)
            val_correct += (preds == Y_batch).sum().item()
            val_total += Y_batch.size(0)

    val_acc: float = val_correct / val_total


    print(f"Epoch {epoch+1} Summary -> Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} || Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}\n")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save the physical weights to your VM
        torch.save(model.state_dict(), 'best_alpha_model.pth')
        print(f"  [*] Val Loss improved. Saved best_alpha_model.pth")

Using device: cuda


Epoch 1/10 [Val  ]: 100%|██████████| 984/984 [00:03<00:00, 266.49it/s]


Epoch 1 Summary -> Train Loss: 0.4002 | Train Acc: 0.5935 || Val Loss: 0.4160 | Val Acc: 0.6429

  [*] Val Loss improved. Saved best_alpha_model.pth


Epoch 2/10 [Val  ]: 100%|██████████| 984/984 [00:03<00:00, 265.51it/s]


Epoch 2 Summary -> Train Loss: 0.3861 | Train Acc: 0.6078 || Val Loss: 0.3915 | Val Acc: 0.5466

  [*] Val Loss improved. Saved best_alpha_model.pth


...

Epoch 8 Summary -> Train Loss: 0.3625 | Train Acc: 0.6414 || Val Loss: 0.4067 | Val Acc: 0.6800



Epoch 9/10 [Val  ]: 100%|██████████| 984/984 [00:03<00:00, 253.26it/s]


Epoch 9 Summary -> Train Loss: 0.3608 | Train Acc: 0.6408 || Val Loss: 0.3949 | Val Acc: 0.5741



Epoch 10/10 [Val  ]: 100%|██████████| 984/984 [00:03<00:00, 259.62it/s]

Epoch 10 Summary -> Train Loss: 0.3566 | Train Acc: 0.6415 || Val Loss: 0.3993 | Val Acc: 0.6009



In [13]:
import numpy as np
from sklearn.metrics import confusion_matrix
from typing import List
import torch
from torch.utils.data import DataLoader

def analyze_thresholds(alpha_model: torch.nn.Module, val_loader: DataLoader, thresholds: List[float]) -> None:
    """
    Evaluates hit rates and confusion matrices across confidence thresholds
    without running the heavy exchange execution simulator.
    """
    alpha_model.eval()
    all_probs_list: List[np.ndarray] = []
    all_targets_list: List[np.ndarray] = []

    with torch.no_grad():
        for X_batch, Y_batch, _ in val_loader:
            logits: torch.Tensor = alpha_model(X_batch.to(device))
            probs: torch.Tensor = torch.nn.functional.softmax(logits, dim=1)

            all_probs_list.append(probs.cpu().numpy())
            all_targets_list.append(Y_batch.numpy())

    # p_matrix shape: [N, 3] representing probabilities for [DOWN, FLAT, UP]
    p_matrix: np.ndarray = np.concatenate(all_probs_list, axis=0)
    y_true: np.ndarray = np.concatenate(all_targets_list, axis=0)

    print("=== MODEL CONFIDENCE AUTOSPY ===")

    for t in thresholds:
        max_probs: np.ndarray = np.max(p_matrix, axis=1)
        valid_idx: np.ndarray = np.where(max_probs >= t)[0]

        if len(valid_idx) == 0:
            print(f"\n[Threshold {t:.2f}] 0 signals fired.")
            continue

        y_pred_t: np.ndarray = np.argmax(p_matrix[valid_idx], axis=1)
        y_true_t: np.ndarray = y_true[valid_idx]

        hit_rate: float = np.mean(y_pred_t == y_true_t)
        cm: np.ndarray = confusion_matrix(y_true_t, y_pred_t, labels=[0, 1, 2])

        print(f"\n[Threshold {t:.2f}] Total Signals: {len(valid_idx)} | Hit Rate: {hit_rate:.2%}")
        print(f"Confusion Matrix (Rows: True, Cols: Pred [DOWN, FLAT, UP]):")
        print(cm)

# --- Execution ---
# Assumes 'loader' is your standard validation DataLoader containing the 14,000 sequences
analyze_thresholds(model, val_loader, thresholds=[0.50, 0.55, 0.60, 0.65, 0.70, 0.75])

=== MODEL CONFIDENCE AUTOSPY ===

[Threshold 0.50] Total Signals: 33986 | Hit Rate: 88.41%
Confusion Matrix (Rows: True, Cols: Pred [DOWN, FLAT, UP]):
[[  824   126   280]
 [ 1682 28678  1408]
 [  342   101   545]]

[Threshold 0.55] Total Signals: 28479 | Hit Rate: 95.57%
Confusion Matrix (Rows: True, Cols: Pred [DOWN, FLAT, UP]):
[[  368    89    69]
 [  577 26680   338]
 [  114    75   169]]

[Threshold 0.60] Total Signals: 25418 | Hit Rate: 98.66%
Confusion Matrix (Rows: True, Cols: Pred [DOWN, FLAT, UP]):
[[   95    71    12]
 [  120 24948    67]
 [   16    55    34]]

[Threshold 0.65] Total Signals: 23531 | Hit Rate: 99.49%
Confusion Matrix (Rows: True, Cols: Pred [DOWN, FLAT, UP]):
[[    3    63     0]
 [   10 23402     8]
 [    0    40     5]]

[Threshold 0.70] Total Signals: 21878 | Hit Rate: 99.63%
Confusion Matrix (Rows: True, Cols: Pred [DOWN, FLAT, UP]):
[[    0    49     0]
 [    1 21797     0]
 [    0    31     0]]

[Threshold 0.75] Total Signals: 20033 | Hit Rate: 99.72%

In [15]:
#THIS IS AN OLDER VERSION, reuires fixes to the class labeling, and may require other changes
#
#

import os
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List

# --- CONFIGURATION ---
VAL_FILES_LIST = 'val_files.txt' # The exact list saved from your training script
MODEL_WEIGHTS_PATH = 'best_alpha_model.pth'
OUTPUT_PREDS_PATH = 'predictions.parquet'
CONFIDENCE_THRESHOLD = 0.65 # Minimum confidence to send to simulator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 2. THE INFERENCE ENGINE ---
def run_inference_engine(model: torch.nn.Module, dataloader: DataLoader):
    model.eval()
    all_predictions = []

    print("Executing forward passes...")
    with torch.no_grad():
        # The DataLoader Contract: must yield ticker, ts, local_ts, X
        for tickers, ts_batch, local_ts_batch, x_batch in dataloader:
            x_batch = x_batch.to(device)

            # Forward pass
            logits = model(x_batch)
            probs = F.softmax(logits, dim=1)

            # Extract max probabilities and the predicted class
            confidences, predicted_classes = torch.max(probs, dim=1)

            # Move to CPU for DataFrame
            confidences = confidences.cpu().numpy()
            predicted_classes = predicted_classes.cpu().numpy()
            ts_batch = ts_batch.numpy()
            local_ts_batch = local_ts_batch.numpy()

            # Assuming Classes: 0 = No, 1 = Yes, 2 = Hold based on your training script
            side_map = {0: 'no', 1: 'yes', 2: 'hold'}

            for i in range(len(tickers)):
                conf = confidences[i]
                pred_side = side_map[predicted_classes[i]]

                # Filter noise: Only log actionable trades above confidence threshold
                if pred_side != 'hold' and conf >= CONFIDENCE_THRESHOLD:
                    all_predictions.append({
                        'ticker': tickers[i],
                        'ts': ts_batch[i],
                        'local_recv_ts': local_ts_batch[i],
                        'predicted_side': pred_side,
                        'confidence': conf
                    })

    df_preds = pd.DataFrame(all_predictions)
    if not df_preds.empty:
        # Sort chronologically by local arrival time to simulate pure forward flow
        df_preds = df_preds.sort_values(by='local_recv_ts').reset_index(drop=True)
        df_preds.to_parquet(OUTPUT_PREDS_PATH, index=False)
        print(f"\n--- Engine Complete ---")
        print(f"Saved {len(df_preds)} actionable signals to {OUTPUT_PREDS_PATH}.")
        print(df_preds.head())
    else:
        print("\n--- Engine Complete ---")
        print(f"No signals met the confidence threshold of {CONFIDENCE_THRESHOLD}.")

if __name__ == '__main__':
    # 1. Load the exact validation files used in training
    if not os.path.exists(VAL_FILES_LIST):
        raise FileNotFoundError(f"Missing {VAL_FILES_LIST}. Please export this from your training script.")

    with open(VAL_FILES_LIST, 'r') as f:
        val_files = [line.strip() for line in f.readlines()]

    # 2. Instantiate Dataset in Inference Mode
    val_dataset = KalshiDenseDataset(val_files, seq_len=60, inference_mode=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    # 3. Instantiate your exact model architecture
    # NOTE: You must copy your KalshiTransformer class definition into this notebook
    # model = KalshiTransformer(feature_dim=23, d_model=64, nhead=4, num_layers=2, num_classes=3).to(device)
    # model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))

    # 4. Run
    run_inference_engine(model, val_loader)

Using device: cuda


FileNotFoundError: Missing val_files.txt. Please export this from your training script.

In [16]:
# --- CONFIGURATION ---
CONFIDENCE_THRESHOLD = 0.55 # Lowered to ensure we get ~1,600 signals for the simulator
OUTPUT_PREDS_PATH = 'predictions.parquet'

def run_inference_engine(model: torch.nn.Module, dataloader: DataLoader):
    # Load the best weights we just saved during the training loop
    model.load_state_dict(torch.load('best_alpha_model.pth', weights_only=True))
    model.eval()
    all_predictions = []

    print(f"Executing forward passes at Threshold: {CONFIDENCE_THRESHOLD}...")
    with torch.no_grad():
        for tickers, ts_batch, local_ts_batch, x_batch in dataloader:
            x_batch = x_batch.to(device)

            logits = model(x_batch)
            probs = F.softmax(logits, dim=1)

            confidences, predicted_classes = torch.max(probs, dim=1)

            confidences = confidences.cpu().numpy()
            predicted_classes = predicted_classes.cpu().numpy()
            ts_batch = ts_batch.numpy()
            local_ts_batch = local_ts_batch.numpy()

            # FIXED MAPPING: 0=DOWN(no), 1=FLAT(hold), 2=UP(yes)
            side_map = {0: 'no', 1: 'hold', 2: 'yes'}

            for i in range(len(tickers)):
                conf = confidences[i]
                pred_side = side_map[predicted_classes[i]]

                if pred_side != 'hold' and conf >= CONFIDENCE_THRESHOLD:
                    all_predictions.append({
                        'ticker': tickers[i],
                        'ts': ts_batch[i],
                        'local_recv_ts': local_ts_batch[i],
                        'predicted_side': pred_side,
                        'confidence': conf
                    })

    df_preds = pd.DataFrame(all_predictions)
    if not df_preds.empty:
        # Sort chronologically by local arrival time to simulate pure forward flow
        df_preds = df_preds.sort_values(by='local_recv_ts').reset_index(drop=True)
        df_preds.to_parquet(OUTPUT_PREDS_PATH, index=False)
        print(f"\n--- Engine Complete ---")
        print(f"Saved {len(df_preds)} actionable signals to {OUTPUT_PREDS_PATH}.")
        print(df_preds.head())
    else:
        print("\n--- Engine Complete ---")
        print(f"No signals met the confidence threshold of {CONFIDENCE_THRESHOLD}.")

if __name__ == '__main__':
    # 1. We just use the `val_files` list that is already in your Colab memory!
    print("Building Inference Dataset...")
    val_dataset_inference = KalshiDenseDataset(val_files, seq_len=60, forward_look=30, threshold=0.02, stride=5, inference_mode=True)
    val_loader_inference = DataLoader(val_dataset_inference, batch_size=256, shuffle=False)

    # 2. `model` is already instantiated in your memory from the training loop, so we just pass it in
    run_inference_engine(model, val_loader_inference)

Building Inference Dataset...
Loading 22 dense files into RAM (Inference Mode: True)...

=== FINAL ENGINEERING AUDIT ===
Total valid 60-step sequences: 62964
Dropped due to NaNs: 0
Executing forward passes at Threshold: 0.55...

--- Engine Complete ---
Saved 1475 actionable signals to predictions.parquet.
                            ticker            ts  local_recv_ts  \
0  KXMLBGAME-26APR151840WSHPIT-WSH  1.776293e+09   1.776293e+09   
1  KXMLBGAME-26APR151840WSHPIT-WSH  1.776293e+09   1.776293e+09   
2  KXMLBGAME-26APR151840WSHPIT-WSH  1.776293e+09   1.776293e+09   
3  KXMLBGAME-26APR151840WSHPIT-WSH  1.776293e+09   1.776293e+09   
4  KXMLBGAME-26APR151840WSHPIT-WSH  1.776293e+09   1.776293e+09   

  predicted_side  confidence  
0            yes    0.583864  
1             no    0.616459  
2             no    0.633773  
3             no    0.637773  
4             no    0.591162  
